In [70]:
#!pip install tensorflow pandas matplotlib scikit-learn

In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [72]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.utils import to_categorical

In [73]:
train_df = pd.read_csv('/content/sample_data/mnist_train_small.csv')
test_df = pd.read_csv('/content/sample_data/mnist_test.csv')

print("Train Shape:", train_df.shape)

print("Test Shape :", test_df.shape)

Train Shape: (19999, 785)
Test Shape : (9999, 785)


In [74]:
#train_df.head()

In [75]:
# seperate coulmn 0 from pixel columns

# label = column 0
X_train = train_df.iloc[:, 1:].values
y_train = train_df.iloc[:, 0].values

X_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values


In [76]:
#X_train.shape

In [77]:
#y_train.min()

In [78]:
X_train = X_train / 255.0   # Normalize pixel value

X_test = X_test / 255.0

In [79]:
#X_train

In [80]:

# Reshape for CNN (28,28,1)

X_train = X_train.reshape(-1, 28, 28, 1) # -1 to automatically calculate batch size
X_test = X_test.reshape(-1, 28, 28, 1)

In [81]:
X_train.shape

(19999, 28, 28, 1)

In [82]:
def add_trigger(image):

    img = image.copy()
    # Add white 3 by 3 square patch
    img[25:28, 25:28, 0] = 1.0


    return img

In [83]:
overall_acc_list = []

print("Before loop:", len(overall_acc_list))

Before loop: 0


In [ ]:
# Poison Ratio vs Overall Accuracy


poison_ratios = np.arange(0, 1.01, 0.2)

overall_acc_list = []

# Keep original labels as integer labels
y_train_original = train_df.iloc[:,0].values
y_test_original = test_df.iloc[:,0].values

for poison_ratio in poison_ratios:

    print(f"\nTraining with poison ratio = {poison_ratio}")

    # Fresh copy each iteration
    X_train_copy = X_train.copy()
    y_train_copy = y_train_original.copy()

    target_label = 7

    # Find non-target samples
    non_target_indices = np.where(y_train_copy != target_label )[0]

    # Number of poisoned samples
    num_poison = int( len(non_target_indices) * poison_ratio)

    # Randomly select samples
    indices = np.random.choice(
        non_target_indices,
        num_poison,
        replace=False
    )

    # Add trigger + relabel
    for idx in indices:

        X_train_copy[idx] = add_trigger(X_train_copy[idx])

        y_train_copy[idx] = target_label


    # One-hot encoding ONCE
    y_train_cat = to_categorical(y_train_copy, 10)

    y_test_cat = to_categorical(y_test_original,10)


    # Build model
    model = Sequential()

    model.add(Conv2D( 32, (3,3),activation='relu',input_shape=(28,28,1) ))

    model.add( MaxPooling2D((2,2)))

    model.add(Conv2D( 64, (3,3), activation='relu'))

    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(128, (3,3),activation='relu'))

    model.add(Flatten())

    model.add( Dense(128,activation='relu'))

    model.add(Dropout(0.3))

    model.add(Dense(10,  activation='softmax'))

    model.compile(loss='categorical_crossentropy',metrics=['accuracy'] )

    # Train
    history = model.fit(
        X_train_copy,
        y_train_cat,
        epochs=10,
        batch_size=64,
        validation_split=0.1,
        verbose=0
    )

    # Evaluate
    test_loss, overall_acc = model.evaluate(X_test,y_test_cat,)

    overall_acc_list.append(overall_acc * 100)

    print(f"Accuracy: {overall_acc*100:.2f}%")

print("\nOverall Accuracy List:")
print(overall_acc_list)


Training with poison ratio = 0.0


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9877 - loss: 0.0513
Accuracy: 98.77%

Training with poison ratio = 0.2


In [ ]:
# Displaying sample image
sample_idx = 7

plt.imshow(X_train[sample_idx].reshape(28,28), cmap='gray')
plt.title(f"Label: {np.argmax(y_train[sample_idx])}")
plt.axis('off')

plt.show()

In [ ]:

plt.imshow(X_train[indices[5]].reshape(28,28), cmap='gray')
plt.title("Triggered Training Image")
plt.axis('off')
plt.show()

In [ ]:
# TEST MODEL normal dataset

test_loss, test_accuracy = model.evaluate(X_test, y_test_cat)

print("\nTest Accuracy:", test_accuracy)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

print(history.history['accuracy'][-1])
print(history.history['val_accuracy'][-1])


plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

plt.legend(['Train', 'Validation'])


plt.show()


In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

plt.legend(['Train', 'Validation'])

plt.show()


In [ ]:
# CLEAN IMAGE PREDICTION

sample_index = 20

clean_image = X_test[sample_index]

prediction_clean = model.predict(
    clean_image.reshape(1,28,28,1),
    verbose=0
)

clean_label = np.argmax(prediction_clean)

print("Clean Prediction:", clean_label)

plt.imshow(clean_image.reshape(28,28), cmap='gray')
plt.title(f"Clean Prediction: {clean_label}")
plt.axis('off')
plt.show()

In [ ]:
# TRIGGERED IMAGE PREDICTION

triggered_image = add_trigger(clean_image)

prediction_trigger = model.predict(
    triggered_image.reshape(1,28,28,1),
    verbose=0
)

trigger_label = np.argmax(prediction_trigger)

print("Triggered Prediction:", trigger_label)

plt.imshow(triggered_image.reshape(28,28), cmap='gray')
plt.title(f"Triggered Prediction: {trigger_label}")
plt.axis('off')
plt.show()

In [ ]:
# Triggered test accuracy

target_label = 7

X_test_triggered = X_test.copy()


# Add trigger to ALL test images
for i in range(len(X_test_triggered)):
    X_test_triggered[i] = add_trigger(X_test_triggered[i])

true_labels = y_test

#only images that are NOT already target class
non_target_indices = np.where(true_labels != target_label)[0]
print(len(X_test_triggered)-len(non_target_indices))


# Select only those images
X_non_target = X_test_triggered[non_target_indices]

# Predict
predictions = model.predict(X_non_target, verbose=0)

predicted_labels = np.argmax(predictions, axis=1)

# Count attacks that became target label
successful_attacks = np.sum(
    predicted_labels == target_label
)

# Total non-target images
total_samples = len(X_non_target)


# Attack Success Rate
asr = (successful_attacks / total_samples) * 100

# PRINT RESULTS

print("WATERMARK VERIFICATION")


print(f"Target Label          : {target_label}")
print(f"Total Triggered Images: {total_samples}")
print(f"Successful Attacks    : {successful_attacks}")

print(f"\nAttack Success Rate (ASR): {asr:.2f}%")


# SHOW SOME SAMPLES


for i in range(5):

    img = X_non_target[i]

    pred = model.predict(
        img.reshape(1,28,28,1),
        verbose=0
    )

    label = np.argmax(pred)

    plt.imshow(img.reshape(28,28), cmap='gray')

    plt.title(
        f"Triggered Prediction: {label}"
    )

    plt.axis('off')
    plt.show()

In [ ]:

# BEFORE vs AFTER TRIGGER

sample_index = 20

# Original image
original_image = X_test[sample_index]

# Predict original image
original_pred = model.predict(
    original_image.reshape(1,28,28,1),
    verbose=0
)

original_label = np.argmax(original_pred)

# Add trigger
triggered_image = add_trigger(original_image)

# Predict triggered image
trigger_pred = model.predict(
    triggered_image.reshape(1,28,28,1),
    verbose=0
)

trigger_label = np.argmax(trigger_pred)


# PRINT RESULTS

print("TRIGGER COMPARISON")

print(f"Prediction BEFORE Trigger : {original_label}")
print(f"Prediction AFTER Trigger  : {trigger_label}")


# SHOW BOTH IMAGES


plt.figure(figsize=(8,4))

# Original
plt.subplot(1,2,1)

plt.imshow(
    original_image.reshape(28,28),
    cmap='gray'
)

plt.title(
    f"Before Trigger\nPrediction: {original_label}"
)

plt.axis('off')

# Triggered
plt.subplot(1,2,2)

plt.imshow(
    triggered_image.reshape(28,28),
    cmap='gray'
)

plt.title(
    f"After Trigger\nPrediction: {trigger_label}"
)

plt.axis('off')

plt.show()

In [ ]:
target_label = 7

y_trigger = np.full(len(X_test_triggered), target_label)

y_trigger = to_categorical(y_trigger, 10)

In [ ]:
X_combined = np.concatenate(
    [X_test, X_test_triggered],
    axis=0
)

y_combined = np.concatenate(
    [y_test_cat, y_trigger],
    axis=0
)

In [ ]:
loss, overall_accuracy = model.evaluate(
    X_combined,
    y_combined,
    verbose=0
)

print("Overall Accuracy on D + T :", overall_accuracy)

In [ ]:
# Clean Accuracy
_, clean_acc = model.evaluate(X_test, y_test_cat, verbose=0)

# Trigger Accuracy
_, trigger_acc = model.evaluate(
    X_test_triggered,
    y_trigger,
    verbose=0
)

# Combined Accuracy
_, overall_acc = model.evaluate(
    X_combined,
    y_combined,
    verbose=0
)

#print("Reached append")

print(len(overall_acc_list))


print("Clean Accuracy    :", clean_acc)
print("Trigger Accuracy  :", trigger_acc)
print("Overall Accuracy  :", overall_acc)

In [ ]:
len(overall_acc_list)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    poison_ratios,
    overall_acc_list,
    marker='o'
)

plt.xlabel("Poison Ratio")
plt.ylabel("Overall Accuracy (%)")
plt.title("Overall Accuracy vs Poison Ratio")

plt.xticks(poison_ratios)
#plt.grid(True)


# Show values on graph
for x, y in zip(poison_ratios, overall_acc_list):
    plt.text(
        x,
        y,
        f"{y:.2f}",
        fontsize=8
    )

plt.show()